In [6]:
from pathlib import Path
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline

# fast API
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


In [ ]:
# --- absolute relative paths ---
# BASE_DIR = Path(__file__).resolve().parent # only for .py files
# BASE_DIR = Path.cwd() # for jupyter notebook

MODELS_DIR = (BASE_DIR.parent / 'models').resolve()

model_path =        Path(os.environ.get("model_path", MODELS_DIR / "xgb_model.joblib"))
preprocessor_path = Path(os.environ.get("preprocessor_path", MODELS_DIR / "xgb_preprocessor.joblib"))
features_path =     Path(os.environ.get("features_path", MODELS_DIR / "features.json"))
features_all_path = Path(os.environ.get("features_all_path", MODELS_DIR / "features_all.json"))


In [8]:
# --- load artifacts once ---

# model_path = '../models/xgb_model.joblib'
# preprocessor_path = '../models/xgb_preprocessor.joblib'
# features_path = '../models/features.json'
# features_all_path = '../models/features_all.json'

threshold = float(0.5)

model = joblib.load(model_path)
preprocessor = joblib.load(preprocessor_path)
feature_names_to_model = json.load(open(features_path, 'r'))
feature_names_all = json.load(open(features_all_path, 'r'))

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [19]:
# --- requests/responce schema with pydantic classes ---

# if needed in reserve
class Transaction_all(BaseModel):
    id: int
    Time: float
    feat1: float
    feat2: float
    feat3: float
    feat4: float
    feat5: float
    feat6: float
    feat7: float
    feat8: float
    feat9: float
    feat10: float
    feat11: float
    feat12: float
    feat13: float
    feat14: float
    feat15: float
    feat16: float
    feat17: float
    feat18: float
    feat19: float
    feat20: float
    feat21: float
    feat22: float
    feat23: float
    feat24: float
    feat25: float
    feat26: float
    feat27: float
    feat28: float
    Transaction_Amount: float
    IsFraud : bool

class Transaction(BaseModel):
    feat1: float
    feat2: float
    feat3: float
    feat4: float
    feat5: float
    feat6: float
    feat7: float
    feat8: float
    feat9: float
    feat10: float
    feat11: float
    feat12: float
    feat13: float
    feat14: float
    feat15: float
    feat16: float
    feat17: float
    feat18: float
    feat19: float
    feat20: float
    feat21: float
    feat22: float
    feat23: float
    feat24: float
    feat25: float
    feat26: float
    feat27: float
    feat28: float
    Transaction_Amount: float

class InferenceRequest(BaseModel):
    records: list[Transaction] = Field(..., min_items=1, description='Transactions to score')

class InferenceResponse(BaseModel):
    probabilities: list[float]
    labels: list[int]


C:\Users\tinew\AppData\Local\Temp\ipykernel_16392\634130794.py:70: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  records: list[Transaction] = Field(..., min_items=1, description='Transactions to score')


In [ ]:
# --- FastAPI app & routes ---
app = FastAPI(title='Creadit Card Fraud Detector', version = '0.1.0')

@app.get('/health')
def health_check() -> dict[str, str]:
    return {'status': 'ok'}

@app.post('/predict/', response_model=InferenceResponse)
def predict(request: InferenceRequest) -> InferenceResponse:
    try:
        df = pd.DataFrame([record.model_dump() for record in request.records])
        df = df[feature_names_to_model] # ensure exact column order
    except KeyError as exc:
        missing = set(feature_names_to_model) - set(df.columns)
        raise HTTPException(status_code=422, detail=f'MIssing features: {missing}') from exc
    
    proba = pipeline.predict_proba(df)[:,1]
    labels = (proba >= threshold).astype(int)

    return InferenceResponse(probabilities=proba.tolist(), labels=labels.tolist())

In [ ]:
# def my_decorator(func):
#     def wrapper(*args, **kwargs):
#         print("Something is happening before the function is called.")
#         result = func(*args, **kwargs)
#         print("Something is happening after the function is called.")
#         return result
#     return wrapper

# @my_decorator
# def say_hello(name):
#     print(f"Hello, {name}!")

# say_hello("Alice")

Something is happening before the function is called.
Hello, Alice!
Something is happening after the function is called.
